In [1]:
import os
from pathlib import Path
import numpy as np
import h5py

print(f'Current working directory: {os.getcwd()}')

Current working directory: c:\Users\Rimehaug\Documents\projects\fUSI\FusiConverter\create_proxy_hq_files


Change Working Directory

In [2]:
if Path().cwd().name == 'create_proxy_hq_files':
    os.chdir(Path().cwd().parent)
print(f'Current working directory: {os.getcwd()}')


Current working directory: c:\Users\Rimehaug\Documents\projects\fUSI\FusiConverter


## Load Data

In [3]:
#session_date_id = '20251211-2'
session_date_id = '20260624-0'

In [4]:
print('Loading data...')

load_data_dir = Path('../data') / session_date_id

paths_mat = sorted([path for path in load_data_dir.glob('*/*.mat') if 'BM' not in path.stem and 'DP' not in path.stem and 'HQ' not in path.stem])

paths_mat

Loading data...


[WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1654_fUS_T_0.mat'),
 WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1655_fUS_T_1.mat'),
 WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1656_fUS_T_2.mat'),
 WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1657_fUS_T_3.mat'),
 WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1658_fUS_T_4.mat'),
 WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1659_fUS_T_5.mat'),
 WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1700_fUS_T_6.mat'),
 WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1701_fUS_T_7.mat'),
 WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1702_fUS_T_8.mat'),
 WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1703_fUS_T_9.mat'),
 WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1704_fUS_T_10.mat'),
 WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1705_fUS_T_11.mat'),
 WindowsPath('../data/2026

#### Create HQ proxy by averaging over trials and frames

In [5]:
I_trial_avg = []

for i in range(len(paths_mat)):
    with h5py.File(paths_mat[i], 'r') as file:
        I = file['I'][()].T
        I_trial_avg.append(I.mean(axis=-1))

I_hq_proxy = np.mean(I_trial_avg,axis=0)
I_hq_proxy.shape

(90, 64, 54)

#### Save HQ proxy created

In [7]:
save_path = paths_mat[0].with_name(paths_mat[0].stem.rsplit("_", 2)[0] + '_HQproxy.mat')

with h5py.File(paths_mat[0], 'r') as src, h5py.File(save_path, 'w') as dst:
    # written transposed, the same way MATLAB stores it, because the loader transposes back with .T
    dst['I'] = I_hq_proxy.astype(np.float32)
    dst['metadata/voxelSize'] = src['metadata/voxelSize'][()]

save_path


WindowsPath('../data/20260624-0/20260624-0/BECU184_20260624_1654_fUS_HQproxy.mat')